## imports

Run the notebook from the repo root (`newdata_set`) so `PROJECT_ROOT` resolves to `rgc_dataset/`, `final_runs/`, and `all_cam_output/` correctly.

**Supervised Grad-CAM:** run the definition cell, then the last code cell (it rescans `./final_runs` and loops over **every** non-BYOL model). Or use **Run All**. The loop is long: each run × 5 folds × ~412 test images.

In [11]:
from __future__ import annotations

import os
import pickle
from pathlib import Path

import albumentations
import cv2
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torchvision
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from torchmetrics.classification import Accuracy, F1Score, Precision, Recall
from torchvision import datasets
from torchvision.models import convnext_base, resnet50, vgg16, vit_b_16, swin_b

PROJECT_ROOT = Path.cwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
import os

models = {}
run_folder_path = "./final_runs"

for model_name in os.listdir(run_folder_path):
    model_path = os.path.join(run_folder_path, model_name)
    if not os.path.isdir(model_path):
        continue

    models[model_name] = []

    # collect fold folders
    folds = [
        f for f in os.listdir(model_path)
        if os.path.isdir(os.path.join(model_path, f))
    ]

    # sort folds **once**
    folds.sort()

    # iterate over sorted folds
    for fold in folds:
        ckpt_dir = os.path.join(model_path, fold, "checkpoints")
        if not os.path.isdir(ckpt_dir):
            continue

        # pick all ckpts (or just latest)
        ckpt_files = sorted([f for f in os.listdir(ckpt_dir) if f.endswith(".ckpt")])
        if not ckpt_files:
            continue

        # pick first or latest checkpoint
        ckpt_path = os.path.join(ckpt_dir, ckpt_files[0])
        models[model_name].append(ckpt_path)

print(models)

{'resnet50-no_spurious': ['./final_runs/resnet50-no_spurious/fold_0_run-20260319_185315-lovbdi1e/checkpoints/epoch=29-step=1560.ckpt', './final_runs/resnet50-no_spurious/fold_1_run-20260319_185641-x20jzfmu/checkpoints/epoch=29-step=1560.ckpt', './final_runs/resnet50-no_spurious/fold_2_run-20260319_185955-w4tkzsyl/checkpoints/epoch=29-step=1560.ckpt', './final_runs/resnet50-no_spurious/fold_3_run-20260319_190317-tgba6q9k/checkpoints/epoch=29-step=1560.ckpt', './final_runs/resnet50-no_spurious/fold_4_run-20260319_190641-h39huxez/checkpoints/epoch=29-step=1560.ckpt'], 'swin_b-spurious': ['./final_runs/swin_b-spurious/fold_0_run-20260319_214602-iyxafk06/checkpoints/epoch=29-step=1560.ckpt', './final_runs/swin_b-spurious/fold_1_run-20260319_215822-1dnyqcu2/checkpoints/epoch=29-step=1560.ckpt', './final_runs/swin_b-spurious/fold_2_run-20260319_221013-5g1tbrsk/checkpoints/epoch=29-step=1560.ckpt', './final_runs/swin_b-spurious/fold_3_run-20260319_222201-xwzswewm/checkpoints/epoch=29-step=1560

In [13]:
# Optional handles (same keys as under ./final_runs/<name>/fold_*/checkpoints/)
convnext_spurious = models["convnext_base-spurious"]
convnext_no_spurious = models["convnext_base-no_spurious"]
resnet50_spurious = models["resnet50-spurious"]
resnet50_no_spurious = models["resnet50-no_spurious"]
vgg16_spurious = models["vgg16-spurious"]
vgg16_no_spurious = models["vgg16-no_spurious"]
vit_b_16_spurious = models["vit_b_16-spurious"]
vit_b_16_no_spurious = models["vit_b_16-no_spurious"]
swin_b_spurious = models["swin_b-spurious"]
swin_b_no_spurious = models["swin_b-no_spurious"]


In [ ]:
# ── Albumentations transform for supervised models (RGB 224×224, ImageNet stats) ──
class TransformsAlbu:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, img):
        return self.transforms(image=np.array(img))["image"]

SUPERVISED_TRANSFORM = TransformsAlbu(albumentations.Compose([
    albumentations.Resize(224, 224),
    albumentations.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
]))


# ── RGB ImageFolder with filename ──
class ImageFolderWithFilename(datasets.ImageFolder):
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        path = self.samples[index][0]
        filename = os.path.basename(path)
        return img, target, filename, path


# ── Model factory (mirrors training script) ──
def select_model(model_name, num_classes):
    if model_name == "resnet50":
        m = resnet50(weights="DEFAULT")
        m.fc = torch.nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "vgg16":
        m = vgg16(weights="DEFAULT")
        m.classifier[6] = torch.nn.Linear(m.classifier[6].in_features, num_classes)
    elif model_name == "convnext_base":
        m = convnext_base(weights="DEFAULT")
        m.classifier[2] = torch.nn.Linear(m.classifier[2].in_features, num_classes)
    elif model_name == "vit_b_16":
        m = vit_b_16(weights="DEFAULT")
        m.heads = torch.nn.Linear(m.heads.head.in_features, num_classes)
    elif model_name == "swin_b":
        m = swin_b(weights="DEFAULT")
        m.head = torch.nn.Linear(m.head.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return m


# ── Lightning wrapper (mirrors training script) ──
class VisionClassifier(pl.LightningModule):
    def __init__(self, model, num_classes, learning_rate=1e-4):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.num_classes = num_classes
        self.criterion = torch.nn.CrossEntropyLoss()
        self.train_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_accuracy   = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_accuracy  = Accuracy(task="multiclass", num_classes=num_classes)
        self.precision = Precision(task="multiclass", num_classes=num_classes)
        self.recall    = Recall(task="multiclass", num_classes=num_classes)
        self.f1_score  = F1Score(task="multiclass", num_classes=num_classes)
        self.test_outputs = []
    def forward(self, x):
        return self.model(x)
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


# ── Grad-CAM target layers per architecture ──
def _vit_gradcam_target_layer(m):
    """
    ViT Grad-CAM works better on the pre-attention norm layer than on the
    final encoder output (which can yield near-zero patch-token gradients).
    """
    last_block = m.model.encoder.layers[-1]
    if hasattr(last_block, "norm1"):
        return last_block.norm1
    if hasattr(last_block, "ln_1"):
        return last_block.ln_1
    # Fallback if layer naming differs across torchvision versions
    return m.model.encoder.layers[-2]


ARCH_TARGET_LAYER = {
    "resnet50":      lambda m: m.model.layer4[-1],        # Bottleneck, (B,C,H,W)
    "vgg16":         lambda m: m.model.features[29],      # last ReLU,  (B,C,H,W)
    "convnext_base": lambda m: m.model.features[-1][-1],  # last CNBlock,(B,C,H,W)
    "vit_b_16":      _vit_gradcam_target_layer,           # ViT norm layer, (B,N+1,C)
    "swin_b":        lambda m: m.model.features[-1][-1],  # SwinBlock,  (B,H,W,C)
}
# How to reshape activations/gradients to (B, C, H, W)
ARCH_RESHAPE = {
    "resnet50":      "none",
    "vgg16":         "none",
    "convnext_base": "none",
    "vit_b_16":      "vit",   # (B, N+1, C) with CLS token
    "swin_b":        "swin",  # (B, H, W, C)
}


# ── Generic Grad-CAM (hook-based, works for all torchvision architectures) ──
def get_cam_generic(model, target_layer, image, target, device, reshape="none"):
    model.eval()

    act_holder, grad_holder = {}, {}

    def fwd_hook(module, inp, out):
        # Store forward activations and capture their gradient directly.
        # This avoids module-level backward hooks that can fail with in-place ops (e.g., VGG ReLU).
        act_holder["v"] = out
        out.register_hook(lambda g: grad_holder.__setitem__("v", g))

    h1 = target_layer.register_forward_hook(fwd_hook)

    image    = image.to(device)
    target_t = target.to(device).long()
    if target_t.dim() == 0:
        target_t = target_t.unsqueeze(0)

    model.zero_grad(set_to_none=True)
    output = model(image)
    probs  = F.softmax(output, dim=1).detach().cpu()
    pred   = torch.argmax(output, dim=1).detach().cpu()

    # Backprop on PREDICTED class (class-discriminative Grad-CAM)
    # logits[:, argmax]  — NOT ground truth, NOT cross-entropy
    pred_class = torch.argmax(output, dim=1)[0]
    score = output[0, pred_class]
    score.backward(retain_graph=True)

    h1.remove()

    act  = act_holder["v"]
    grad = grad_holder["v"]

    if reshape == "vit":
        # (B, N+1, C)  N+1 includes CLS token at position 0
        B, N1, C = act.shape
        n_patches = N1 - 1
        H = W = int(n_patches ** 0.5)
        act  = act[:, 1:, :].reshape(B, H, W, C).permute(0, 3, 1, 2)
        grad = grad[:, 1:, :].reshape(B, H, W, C).permute(0, 3, 1, 2)
    elif reshape == "swin":
        # (B, H, W, C)
        act  = act.permute(0, 3, 1, 2)
        grad = grad.permute(0, 3, 1, 2)
    # else "none": already (B, C, H, W)

    pooled_grad = torch.mean(grad, dim=[0, 2, 3])
    activations = act.detach() * pooled_grad.view(1, -1, 1, 1)

    heatmap = torch.mean(activations, dim=1).squeeze()
    heatmap = torch.relu(heatmap).cpu().numpy()

    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()

    # Small maps (e.g. 7×7 for ConvNeXt at 224×224): INTER_CUBIC rings at the frame and can
    # overshoot outside [0,1]; uint8(255*x) then wraps and looks like spurious hot borders.
    heatmap_raw = heatmap.copy()
    heatmap_2d = cv2.resize(
        heatmap.astype(np.float32), (150, 150), interpolation=cv2.INTER_LINEAR
    )
    heatmap_2d = np.clip(heatmap_2d, 0.0, 1.0)
    heatmap_uint8 = np.uint8(255 * heatmap_2d)

    threshold_value = int(0.01 * 255)
    _, thresh = cv2.threshold(heatmap_uint8, threshold_value, 255, cv2.THRESH_BINARY)
    cnt_result = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = cnt_result[0] if len(cnt_result) == 2 else cnt_result[1]

    heatmap_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    # Same order as BYOL get_cam: native map, JET BGR 150², contours, pred, float 150², probs
    return heatmap_raw, heatmap_bgr, contours, pred, heatmap_2d, probs


print("Non-BYOL definitions loaded.")

Non-BYOL definitions loaded.


: 

In [ ]:
import os
import gc
import pickle
import cv2
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ------------------- Settings -------------------
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "all_cam_output", "supervised_gradcam")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Rescan ./final_runs here so this cell processes *all* supervised models even if the separate scan cell above was not run
models = {}
_run_root = "./final_runs"
for model_name in os.listdir(_run_root):
    model_path = os.path.join(_run_root, model_name)
    if not os.path.isdir(model_path):
        continue
    models[model_name] = []
    folds = [f for f in os.listdir(model_path) if os.path.isdir(os.path.join(model_path, f))]
    folds.sort()
    for fold in folds:
        ckpt_dir = os.path.join(model_path, fold, "checkpoints")
        if not os.path.isdir(ckpt_dir):
            continue
        ckpt_files = sorted([f for f in os.listdir(ckpt_dir) if f.endswith(".ckpt")])
        if not ckpt_files:
            continue
        ckpt_path = os.path.join(ckpt_dir, ckpt_files[0])
        models[model_name].append(ckpt_path)

KNOWN_ARCHS = set(ARCH_TARGET_LAYER.keys())
# Include every non-BYOL run; use explicit order (not alphabetical) so it is obvious all archs are queued
_CANONICAL_ORDER = (
    "resnet50-no_spurious",
    "resnet50-spurious",
    "vgg16-no_spurious",
    "vgg16-spurious",
    "convnext_base-no_spurious",
    "convnext_base-spurious",
    "swin_b-no_spurious",
    "swin_b-spurious",
    "vit_b_16-no_spurious",
    "vit_b_16-spurious",
)
_non_byol = {k for k in models if not k.startswith("byol")}
supervised_run_keys = [k for k in _CANONICAL_ORDER if k in _non_byol]
for k in sorted(_non_byol):
    if k not in supervised_run_keys:
        supervised_run_keys.append(k)

print(
    f"[supervised CAM] {_run_root}: {len(models)} run folder(s), "
    f"{len(supervised_run_keys)} supervised job(s) (all non-BYOL)"
)
for k in supervised_run_keys:
    print(f"  {k}: {len(models[k])} fold ckpt(s)")

skipped_supervised = []

# ------------------- Utility functions -------------------
def parse_run_key(run_key: str):
    arch_name, variant = run_key.rsplit("-", 1)
    if variant not in ("spurious", "no_spurious"):
        return None, None
    return arch_name, variant

def get_data_dir_for_variant(variant: str) -> str:
    if variant == "no_spurious":
        return str(PROJECT_ROOT / "rgc_dataset" / "no_spurious_masked_png")
    return str(PROJECT_ROOT / "rgc_dataset" / "spurious_masked_png")

def stratified_kfold_indices(dataset):
    """5-fold stratified split matching training / all_cam (random_state=42)."""
    targets = [dataset.samples[i][1] for i in range(len(dataset))]
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return list(skf.split(range(len(dataset)), targets))

# ------------------- Main supervised loop -------------------
for run_key in supervised_run_keys:
    arch_name, variant = parse_run_key(run_key)
    print(f"\n{'='*60}\nRun: {run_key}  Arch: {arch_name}  Variant: {variant}")

    if arch_name is None or arch_name not in KNOWN_ARCHS:
        skipped_supervised.append((run_key, "Unknown or unparsable arch"))
        continue

    ckpt_paths = models[run_key]
    if not ckpt_paths:
        skipped_supervised.append((run_key, "No checkpoints found"))
        continue

    data_dir_sv = get_data_dir_for_variant(variant)
    print(f"  Dataset directory: {data_dir_sv}")

    try:
        full_sv_dataset = ImageFolderWithFilename(root=data_dir_sv, transform=SUPERVISED_TRANSFORM)
    except Exception as e:
        skipped_supervised.append((run_key, f"Dataset load error: {e}"))
        continue

    all_fold_indices = stratified_kfold_indices(full_sv_dataset)
    n_folds = len(all_fold_indices)
    if len(ckpt_paths) != n_folds:
        skipped_supervised.append(
            (run_key, f"Need {n_folds} checkpoints (one per fold), found {len(ckpt_paths)}")
        )
        continue

    sv_num_classes = len(full_sv_dataset.classes)
    sv_class_names = full_sv_dataset.classes

    for fold_idx, ckpt_path in enumerate(ckpt_paths):
        print(f"\n  --- fold {fold_idx} ---")
        print(f"  Checkpoint: {ckpt_path}")

        try:
            base_model = select_model(arch_name, sv_num_classes)
            vision_clf = VisionClassifier.load_from_checkpoint(
                ckpt_path,
                model=base_model,
                num_classes=sv_num_classes,
                map_location=device,
                strict=True,
            )
            vision_clf = vision_clf.to(device)
            vision_clf.eval()
        except Exception as e:
            skipped_supervised.append((f"{run_key}_fold_{fold_idx}", str(e)))
            print(f"  [skip fold {fold_idx}] {e}")
            continue

        target_layer = ARCH_TARGET_LAYER[arch_name](vision_clf)
        reshape_mode = ARCH_RESHAPE[arch_name]

        _, test_idx = all_fold_indices[fold_idx]
        sv_dataset = torch.utils.data.Subset(full_sv_dataset, test_idx)
        print(
            f"  Fold-{fold_idx} test split: {len(sv_dataset)}/{len(full_sv_dataset)} samples"
        )

        GRAD_CAM_DIR = os.path.join(
            OUTPUT_ROOT, f"{run_key}_fold_{fold_idx}", "grad_cam_with_sp"
        )
        for sub in (
            "heatmap",
            "heatmap_gray_150",
            "result",
            "image",
            "contour",
            "contour_levels",
            "cubehelix_overlay",
            "plot_data",
            "cam_npz",
            "contours_pkl",
        ):
            os.makedirs(os.path.join(GRAD_CAM_DIR, sub), exist_ok=True)

        preds = []
        heatmaps_raw_dict = {}

        # ------------------- Process each sample -------------------
        for i, data in enumerate(sv_dataset):
            image_t, target_idx, filename, path = data
            image_batch = image_t.unsqueeze(0)
            target_t = torch.tensor([target_idx])

            try:
                heatmap_native, heatmap_bgr, contours, pred, heatmap_2d, probs_tensor = get_cam_generic(
                    vision_clf, target_layer, image_batch, target_t, device, reshape=reshape_mode
                )
            except Exception as e:
                print(f"    [CAM error on {filename}] {e}")
                continue

            stem = os.path.splitext(filename)[0]
            cv2.imwrite(os.path.join(GRAD_CAM_DIR, "heatmap", filename), heatmap_bgr)
            gray150 = np.uint8(255 * np.clip(heatmap_2d.astype(np.float64), 0.0, 1.0))
            cv2.imwrite(os.path.join(GRAD_CAM_DIR, "heatmap_gray_150", filename), gray150)
            image_gray = np.array(Image.open(path).convert("L"))
            image_np = np.array(Image.open(path).convert("RGB"))
            np.savez_compressed(
                os.path.join(GRAD_CAM_DIR, "cam_npz", f"{stem}.npz"),
                heatmap_native=np.asarray(heatmap_native, dtype=np.float32),
                heatmap_150=np.asarray(heatmap_2d, dtype=np.float32),
                pred=np.int64(int(pred.squeeze().item())),
                probs=np.asarray(probs_tensor.squeeze(0).numpy(), dtype=np.float32),
                image_rgb=np.asarray(image_np, dtype=np.uint8),
            )
            with open(os.path.join(GRAD_CAM_DIR, "contours_pkl", f"{stem}.pkl"), "wb") as _cf:
                pickle.dump(contours, _cf)

            print(f"{filename}: Original image shape: {image_np.shape}")

            # Resize for saving
            image_resized = cv2.resize(image_np, (150, 150), interpolation=cv2.INTER_LINEAR)
            cv2.imwrite(os.path.join(GRAD_CAM_DIR, "image", filename), image_resized)

            # ------------------- Grad-CAM overlay -------------------
            gradcam_overlay = heatmap_bgr * 0.3 + image_resized * 0.5
            cv2.imwrite(os.path.join(GRAD_CAM_DIR, "result", filename), gradcam_overlay.astype(np.uint8))

            # ------------------- Binary contour -------------------
            img_contour = image_resized.copy()
            cv2.drawContours(img_contour, contours, -1, (255, 255, 255), 1)
            cv2.imwrite(os.path.join(GRAD_CAM_DIR, "contour", filename), img_contour)
            print(f"{filename}: Contour map shape: {img_contour.shape}")

            # ------------------- Contour levels (Matplotlib) -------------------
            fig, ax = plt.subplots(figsize=(1.5, 1.5), dpi=100)
            ax.imshow(cv2.cvtColor(image_resized, cv2.COLOR_BGR2GRAY), cmap="gray")
            levels = np.unique(np.linspace(heatmap_2d.min(), heatmap_2d.max(), 20))
            if levels.size >= 2:
                ax.contour(heatmap_2d, levels=levels, colors="k", alpha=0.5, linewidths=0.5)
            ax.axis("off")
            plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
            plt.savefig(os.path.join(GRAD_CAM_DIR, "contour_levels", filename))
            plt.close(fig)

            # ------------------- Cubehelix overlay (grayscale) -------------------
            fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
            H_hr, W_hr = 300, 300

            # Resize heatmap to high-res
            heatmap_hr = cv2.resize(heatmap_2d.astype(np.float32), (W_hr, H_hr), interpolation=cv2.INTER_LINEAR)
            heatmap_hr = np.clip(heatmap_hr, 0.0, 1.0)

            # Resize original image to high-res
            image_gray = cv2.resize(image_gray, (W_hr, H_hr), interpolation=cv2.INTER_LINEAR)

            ax.imshow(image_gray, cmap="cubehelix_r")
            levels = np.linspace(heatmap_hr.min()*2+0.3, heatmap_hr.max(), 10)
            levels = np.unique(levels)
            if levels.size >= 2:
                ax.contour(heatmap_hr, levels=levels, colors="black", linewidths=0.65)
            ax.axis("off")
            plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
            plt.savefig(
                os.path.join(GRAD_CAM_DIR, "cubehelix_overlay", filename), dpi=300
            )
            plt.close(fig)

            heatmaps_raw_dict[filename] = heatmap_2d.copy()
            pr = probs_tensor.squeeze(0).numpy()
            preds.append([int(pred.squeeze().item()), target_idx, filename, *pr.tolist()])

        # ------------------- Save heatmaps and predictions -------------------
        with open(os.path.join(GRAD_CAM_DIR, "plot_data", "heatmaps_raw.pkl"), "wb") as f:
            pickle.dump(heatmaps_raw_dict, f)

        with open(os.path.join(GRAD_CAM_DIR, "predictions.csv"), "w") as f:
            prob_cols = [f"prob_{name}" for name in sv_class_names]
            header = ["pred", "ground", "filename", *prob_cols, "pred_name", "ground_name"]
            f.write(",".join(header) + "\n")
            for row in preds:
                pred_idx, ground_idx, fname, *probs = row
                prob_str = ",".join(str(p) for p in probs)
                f.write(f"{int(pred_idx)},{ground_idx},{fname},{prob_str},{sv_class_names[int(pred_idx)]},{sv_class_names[ground_idx]}\n")

        del vision_clf
        torch.cuda.empty_cache()
        print(f"  Saved fold {fold_idx}: {len(preds)} samples → {GRAD_CAM_DIR}")

# ------------------- Log skipped runs -------------------
if skipped_supervised:
    skip_log = os.path.join(OUTPUT_ROOT, "skipped_supervised_models.txt")
    with open(skip_log, "w") as f:
        for name, err in skipped_supervised:
            f.write(f"{name}\n{err}\n{'-'*80}\n")
    print(f"\nSkipped {len(skipped_supervised)} supervised checkpoint(s). Log: {skip_log}")

print(f"\nDone. All outputs under: {OUTPUT_ROOT}")


[supervised CAM] ./final_runs: 12 run folder(s), 10 supervised job(s) (all non-BYOL)
  resnet50-no_spurious: 5 fold ckpt(s)
  resnet50-spurious: 5 fold ckpt(s)
  vgg16-no_spurious: 5 fold ckpt(s)
  vgg16-spurious: 5 fold ckpt(s)
  convnext_base-no_spurious: 5 fold ckpt(s)
  convnext_base-spurious: 5 fold ckpt(s)
  swin_b-no_spurious: 5 fold ckpt(s)
  swin_b-spurious: 5 fold ckpt(s)
  vit_b_16-no_spurious: 5 fold ckpt(s)
  vit_b_16-spurious: 5 fold ckpt(s)

Run: resnet50-no_spurious  Arch: resnet50  Variant: no_spurious
  Dataset directory: /path/to/rgc/rgc_dataset/no_spurious_masked_png

  --- fold 0 ---
  Checkpoint: ./final_runs/resnet50-no_spurious/fold_0_run-20260319_185315-lovbdi1e/checkpoints/epoch=29-step=1560.ckpt


/opt/anaconda3/envs/rgc_new/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.5.2


  Fold-0 test split: 412/2060 samples
nat_L0014_000703.97+092110.7_masked.png: Original image shape: (150, 150, 3)
nat_L0014_000703.97+092110.7_masked.png: Contour map shape: (150, 150, 3)
nat_L0024_001248.72-060652.7_masked.png: Original image shape: (150, 150, 3)
nat_L0024_001248.72-060652.7_masked.png: Contour map shape: (150, 150, 3)
nat_L0043_002346.82+071759.6_masked.png: Original image shape: (150, 150, 3)
nat_L0043_002346.82+071759.6_masked.png: Contour map shape: (150, 150, 3)
nat_L0062_003133.42+013024.0_masked.png: Original image shape: (150, 150, 3)
nat_L0062_003133.42+013024.0_masked.png: Contour map shape: (150, 150, 3)
nat_L0108_005451.31+070307.1_masked.png: Original image shape: (150, 150, 3)
nat_L0108_005451.31+070307.1_masked.png: Contour map shape: (150, 150, 3)
nat_L0112_005601.14-012044.4_masked.png: Original image shape: (150, 150, 3)
nat_L0112_005601.14-012044.4_masked.png: Contour map shape: (150, 150, 3)
nat_L0250_015504.36+043830.0_masked.png: Original image 

/opt/anaconda3/envs/rgc_new/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.1, which is newer than your current Lightning version: v2.5.2


  Fold-0 test split: 412/2060 samples
nat_L0014_000703.97+092110.7_masked.png: Original image shape: (150, 150, 3)
nat_L0014_000703.97+092110.7_masked.png: Contour map shape: (150, 150, 3)
nat_L0024_001248.72-060652.7_masked.png: Original image shape: (150, 150, 3)
nat_L0024_001248.72-060652.7_masked.png: Contour map shape: (150, 150, 3)
nat_L0043_002346.82+071759.6_masked.png: Original image shape: (150, 150, 3)
nat_L0043_002346.82+071759.6_masked.png: Contour map shape: (150, 150, 3)
nat_L0062_003133.42+013024.0_masked.png: Original image shape: (150, 150, 3)
nat_L0062_003133.42+013024.0_masked.png: Contour map shape: (150, 150, 3)
nat_L0108_005451.31+070307.1_masked.png: Original image shape: (150, 150, 3)
nat_L0108_005451.31+070307.1_masked.png: Contour map shape: (150, 150, 3)
nat_L0112_005601.14-012044.4_masked.png: Original image shape: (150, 150, 3)
nat_L0112_005601.14-012044.4_masked.png: Contour map shape: (150, 150, 3)
nat_L0250_015504.36+043830.0_masked.png: Original image 